# Tool Use & Agentes — El Agent Loop

## Módulo 3 — Tool Use | Semana 6

---

## Objetivos de Aprendizaje

Al finalizar esta lección serás capaz de:

| # | Objetivo |
|---|----------|
| 1 | Entender el **patrón ReAct** (Reason + Act) y cómo Claude alterna entre razonar y actuar |
| 2 | Implementar un **agent loop completo** y robusto |
| 3 | Diseñar **condiciones de parada** para evitar loops infinitos y costos descontrolados |
| 4 | **Debuggear agentes** con logging estructurado y trazas |
| 5 | Construir un `AgentResult` que capture métricas completas de ejecución |

---

> **Contexto**: En el ejercicio 5.2 ya construiste un proto-agent loop con `while True`. Ahora vamos a formalizarlo, hacerlo robusto, y entender los patrones que se usan en producción.

---

## 6.1 Patron ReAct (Reason + Act)

**ReAct** viene de "**Rea**soning + **Act**ing". Es el patron fundamental de los agentes basados en LLMs. El agente alterna entre **pensar** y **actuar** hasta que tiene suficiente informacion para responder:

```
Reason -> Act -> Observe -> Reason -> Act -> Observe -> ... -> Respond
```

### Como se ve en la practica con Claude?

```
Step 1: Claude RAZONA  -- "El usuario quiere saber que crags hay cerca de Puebla.
                          Necesito consultar la base de datos con las coordenadas."
        Claude ACTUA   -- llama search_crags(lat=19.04, lon=-98.20, radius=50)

Step 2: Claude OBSERVA -- recibe [{name: "El Cirio", distance: 12km}, ...]
        Claude RAZONA  -- "Encontre 3 crags. Pero el usuario tambien pregunto
                          cuantas rutas tiene cada uno. Necesito otra consulta."
        Claude ACTUA   -- llama get_route_count(crag_id="cirio_01")

Step 3: Claude OBSERVA -- recibe {count: 47}
        Claude RAZONA  -- "Ya tengo toda la informacion. Puedo responder."
        Claude RESPONDE -- "Hay 3 crags cerca de Puebla. El Cirio tiene 47 rutas..."
```

La clave es que **Claude decide en cada step** si necesita mas informacion o si ya puede responder. No es un pipeline fijo -- es **dinamico**.

### Diferencia con Chains

| Aspecto | Chain | Agent (ReAct) |
|---------|-------|---------------|
| **Flujo** | Fijo: paso 1 -> paso 2 -> paso 3 | Dinamico: Claude decide el siguiente paso |
| **Numero de pasos** | Predefinido por tu codigo | Variable, depende del problema |
| **Decision** | Tu codigo decide que ejecutar | Claude decide que tool usar |
| **Cuando parar** | Cuando termina el ultimo paso | Cuando Claude decide que tiene suficiente info |
| **Complejidad** | Baja -- facil de predecir y debuggear | Alta -- el flujo es emergente |

### El Agent Loop en codigo

Esta es la estructura base. Es casi identica a lo que ya tienes en el ejercicio 5.2 -- la diferencia es que ahora vamos a hacerla mas robusta:

In [ ]:
async def agent_loop(user_message: str, tools: list, max_iterations: int = 10):
    """El agent loop completo."""
    messages = [{"role": "user", "content": user_message}]

    for iteration in range(max_iterations):
        # Claude razona y decide
        response = await client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=4096,
            tools=tools,
            messages=messages,
        )

        # Termino? -> respuesta final
        if response.stop_reason == "end_turn":
            return extract_text(response)

        # Quiere tools? -> ejecutar y continuar
        if response.stop_reason == "tool_use":
            # Agregar respuesta de Claude (razonamiento + tool calls)
            messages.append({
                "role": "assistant",
                "content": response.content
            })

            # Ejecutar TODAS las tool calls
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })

            # Devolver resultados (Claude los "observa")
            messages.append({"role": "user", "content": tool_results})

    # Si llegamos aqui, excedimos max_iterations
    return "No pude resolver la tarea en el numero maximo de pasos."

---

## 6.2 Anatomia del Agent Loop

Vamos a desglosar cada componente en detalle.

### El historial de mensajes

El array `messages` crece con cada iteracion. Cada step agrega **2 mensajes**: la respuesta de Claude (`assistant`) y los resultados de tools (`user`). Esto significa que el **costo de tokens crece con cada step** porque Claude recibe todo el historial.

In [ ]:
# Inicio
messages = [
    {"role": "user", "content": "Que crags hay cerca de Puebla?"}
]

# Despues de Step 1 (Claude razona + llama tool)
messages = [
    {"role": "user", "content": "Que crags hay cerca de Puebla?"},
    {"role": "assistant", "content": [
        # TextBlock("Voy a buscar crags cerca de Puebla..."),
        # ToolUseBlock(name="search_crags", input={...}, id="toolu_01")
    ]},
    {"role": "user", "content": [
        {"type": "tool_result", "tool_use_id": "toolu_01", "content": "[...]"}
    ]},
]

# Despues de Step 2 (Claude razona de nuevo + llama otra tool)
messages = [
    # ... todo lo anterior ...
    {"role": "assistant", "content": [
        # TextBlock("Encontre 3 crags. Voy a consultar las rutas..."),
        # ToolUseBlock(name="get_routes", input={...}, id="toolu_02")
    ]},
    {"role": "user", "content": [
        {"type": "tool_result", "tool_use_id": "toolu_02", "content": "[...]"}
    ]},
]

# Step 3 -- Claude responde sin tools
messages = [
    # ... todo lo anterior ...
    {"role": "assistant", "content": [
        # TextBlock("Hay 3 crags cerca de Puebla: ...")
    ]},
]

### Crecimiento de tokens por iteracion

Es importante tener en mente este crecimiento:

| Step | Mensajes en el array | Tokens aprox. (acumulados) |
|------|---------------------|---------------------------|
| 1 | 1 (user) | ~50 |
| 2 | 3 (user + assistant + tool_result) | ~250 |
| 3 | 5 | ~500 |
| N | 2N - 1 | Crece linealmente |

> **Implicacion de costo**: si tu agente toma 5 steps, en el step 5 Claude esta procesando **todo el historial** de los 4 steps anteriores. Esto es input tokens que pagas. Por eso las condiciones de parada son criticas.

### Parsing de tool calls

`response.content` puede contener **multiples bloques mezclados** -- texto (razonamiento) y tool calls:

In [ ]:
# response.content podria ser:
# [
#     TextBlock(type="text", text="Voy a consultar..."),      # razonamiento
#     ToolUseBlock(type="tool_use", name="get_weather",       # tool call 1
#                  input={"city": "Puebla"}, id="toolu_01"),
#     ToolUseBlock(type="tool_use", name="get_time",          # tool call 2
#                  input={"timezone": "Tokyo"}, id="toolu_02"),
# ]

# Necesitas iterar sobre todos:
for block in response.content:
    if block.type == "text":
        # Razonamiento de Claude -- logear para debugging
        logger.info(f"Thinking: {block.text}")
    elif block.type == "tool_use":
        # Tool call -- ejecutar
        result = execute_tool(block.name, block.input)

### Tool result format

Cada resultado **debe** tener el `tool_use_id` correcto que coincida con el `id` del `ToolUseBlock`. Si no coinciden, la API da error:

In [ ]:
# Resultado exitoso
{
    "type": "tool_result",
    "tool_use_id": "toolu_01",  # DEBE coincidir con el id del tool_use
    "content": "resultado como string"
}

# Si la tool fallo, puedes indicarlo:
{
    "type": "tool_result",
    "tool_use_id": "toolu_01",
    "content": "Error: city not found",
    "is_error": True  # Claude sabe que fallo y puede intentar otra cosa
}

El flag `is_error: True` es importante -- le dice a Claude que la tool fallo, y Claude puede decidir:
- Intentar con **otros parametros** (ej. corregir un typo en el nombre de la ciudad)
- Usar **otra tool** diferente
- **Informar al usuario** que no pudo completar esa parte

Sin `is_error`, Claude podria interpretar el mensaje de error como un resultado valido.

### Funcion execute_tool robusta

El ejecutor de tools necesita: **whitelist** (solo tools permitidas), **rate limiting**, y **error handling**:

In [ ]:
def execute_tool(name: str, input_data: dict) -> dict:
    """Ejecuta una tool con whitelist, validacion, y error handling."""

    # 1. Whitelist -- solo tools registradas
    if name not in TOOL_FUNCTIONS:
        return {
            "error": f"Tool '{name}' no existe",
            "available_tools": list(TOOL_FUNCTIONS.keys())
        }

    # 2. Rate limiting
    if not check_rate_limit(name):
        return {"error": f"Rate limit alcanzado para '{name}'"}

    # 3. Ejecutar con manejo de errores
    func = TOOL_FUNCTIONS[name]
    try:
        result = func(**input_data)
        return result
    except TypeError as e:
        # Parametros incorrectos (Claude paso args que la funcion no espera)
        return {"error": f"Parametros invalidos: {e}"}
    except TimeoutError:
        return {"error": f"Timeout ejecutando '{name}'"}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}

---

## 6.3 Condiciones de Parada

Un agente sin condiciones de parada puede correr infinitamente y costarte mucho dinero. Necesitas **multiples mecanismos** de proteccion, no solo uno:

| # | Condicion | Que protege |
|---|-----------|-------------|
| 1 | **Natural stop** | Claude decide que termino (`end_turn`) |
| 2 | **Max iterations** | Limite duro de pasos |
| 3 | **Max token budget** | Limite de costo |
| 4 | **Timeout** | Limite de tiempo |
| 5 | **Error threshold** | Demasiados errores consecutivos |
| 6 | **Loop detection** | Llamadas repetidas con mismos parametros |

> **Regla de produccion**: implementa **al menos 3** de estas. En produccion real, implementa todas.

### 1. Natural stop -- Claude decide que termino

Este es el caso ideal. Claude tiene toda la info y genera la respuesta final:

In [ ]:
if response.stop_reason == "end_turn":
    return extract_text(response)

### 2. Max iterations

Un limite duro que evita loops infinitos. Si se alcanza, **forzamos** a Claude a dar su mejor respuesta con la info que tiene:

In [ ]:
MAX_ITERATIONS = 10

for iteration in range(MAX_ITERATIONS):
    response = await call_claude(messages)
    if response.stop_reason == "end_turn":
        return extract_text(response)
    # ... process tools ...

# Si llegamos aqui, forzar respuesta
messages.append({
    "role": "user",
    "content": "Has alcanzado el limite de pasos. Da tu mejor respuesta "
               "con la informacion que tienes hasta ahora."
})
final = await call_claude(messages)
return extract_text(final)

### 3. Max token budget

Controla el **costo maximo** de una ejecucion. Util en produccion donde cada request tiene un presupuesto:

In [ ]:
total_tokens = 0
MAX_TOKEN_BUDGET = 50_000

for iteration in range(MAX_ITERATIONS):
    response = await call_claude(messages)
    total_tokens += response.usage.input_tokens + response.usage.output_tokens

    if total_tokens > MAX_TOKEN_BUDGET:
        logger.warning(f"Token budget excedido: {total_tokens}")
        break
    # ... process tools ...

### 4. Timeout

Evita que el agente corra indefinidamente esperando respuestas lentas de APIs o del LLM:

In [ ]:
import asyncio

async def agent_with_timeout(message: str, timeout_seconds: int = 60):
    try:
        result = await asyncio.wait_for(
            agent_loop(message),
            timeout=timeout_seconds
        )
        return result
    except asyncio.TimeoutError:
        return "Timeout: la tarea tomo demasiado tiempo."

### 5. Error threshold

Si las tools fallan repetidamente, no tiene sentido seguir intentando. Trackea errores **consecutivos** y aborta si superas un umbral:

In [ ]:
consecutive_errors = 0
MAX_CONSECUTIVE_ERRORS = 3

for iteration in range(MAX_ITERATIONS):
    # ... tool execution ...

    for block in response.content:
        if block.type == "tool_use":
            result = execute_tool(block.name, block.input)

            if "error" in result:
                consecutive_errors += 1
                if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                    return "Demasiados errores consecutivos. Abortando."
            else:
                consecutive_errors = 0  # reset en exito

### 6. Loop detection -- detectar llamadas repetidas

A veces Claude entra en un loop donde llama la misma tool con los mismos parametros una y otra vez. Esto es desperdicio puro de tokens. La solucion: guardar un historial de calls y detectar duplicados:

In [ ]:
call_history = []

for iteration in range(MAX_ITERATIONS):
    # ... get response ...

    for block in response.content:
        if block.type == "tool_use":
            # Crear firma unica: nombre + parametros serializados
            call_signature = f"{block.name}:{json.dumps(block.input, sort_keys=True)}"

            if call_signature in call_history:
                logger.warning(f"Loop detectado: {call_signature}")
                # Forzar a Claude a responder sin tools
                messages.append({
                    "role": "user",
                    "content": "Estas repitiendo la misma llamada. "
                               "Responde con la informacion que ya tienes."
                })
                break

            call_history.append(call_signature)

> **Nota sobre `sort_keys=True`**: Es critico para la deteccion de loops. Sin esto, `{"city": "Puebla", "radius": 50}` y `{"radius": 50, "city": "Puebla"}` serian firmas diferentes aunque sean la misma llamada.

---

## 6.4 Debugging de Agentes

Los agentes son mas dificiles de debuggear que los chains porque el **flujo es dinamico** -- no sabes de antemano cuantos steps va a tomar ni que tools va a usar. Necesitas herramientas para entender que esta pasando.

### Verbose logging -- la traza del agente

La herramienta mas basica y util: loguear cada step con razonamiento, tool calls, y resultados:

In [ ]:
import json

def log_agent_step(iteration: int, response, tool_results: list = None):
    """Pretty print de un step del agente."""
    print(f"\n{'--'*30}")
    print(f"  Step {iteration}")
    print(f"{'--'*30}")

    for block in response.content:
        if block.type == "text" and block.text.strip():
            print(f"  Thinking: {block.text[:200]}")
        elif block.type == "tool_use":
            print(f"  Tool: {block.name}({json.dumps(block.input)})")

    if tool_results:
        for tr in tool_results:
            content = tr["content"]
            if len(content) > 200:
                content = content[:200] + "..."
            print(f"  Result: {content}")

    print(f"  Stop: {response.stop_reason}")
    print(f"  Tokens: in={response.usage.input_tokens}, "
          f"out={response.usage.output_tokens}")

### Ejemplo de traza

Asi se veria la traza de un agente que busca crags cerca de Puebla:

```
------------------------------------------------------------
  Step 1
------------------------------------------------------------
  Thinking: I need to find crags near Puebla. Let me search...
  Tool: search_crags({"lat": 19.04, "lon": -98.20, "radius": 50})
  Result: [{"name": "El Cirio", "distance_km": 12}, ...]
  Stop: tool_use
  Tokens: in=450, out=120

------------------------------------------------------------
  Step 2
------------------------------------------------------------
  Thinking: Found 3 crags. Now I need route counts for each...
  Tool: get_route_count({"crag_id": "cirio_01"})
  Tool: get_route_count({"crag_id": "tepoztlan_01"})
  Result: {"count": 47}
  Result: {"count": 23}
  Stop: tool_use
  Tokens: in=890, out=95

------------------------------------------------------------
  Step 3
------------------------------------------------------------
  Thinking: I have all the information to answer.
  Stop: end_turn
  Tokens: in=1200, out=250
```

Observa como los **input tokens crecen** en cada step: 450 -> 890 -> 1200. Esto es porque Claude recibe todo el historial acumulado.

### Failure modes comunes

Estos son los problemas mas frecuentes al debuggear agentes y como detectarlos:

| Failure Mode | Que pasa | Como detectarlo |
|-------------|----------|-----------------|
| **Hallucination de tool names** | Claude inventa `search_google` que no existe | Whitelist en `execute_tool` lo atrapa |
| **Parametros incorrectos** | Claude pasa `{"ciudad": "Puebla"}` en vez de `{"city": "Puebla"}` | `TypeError` en la funcion, devuelves error, Claude corrige |
| **Loop infinito** | Claude llama la misma tool con los mismos parametros | Loop detection con `call_history` |
| **Ignorar resultados** | Claude llama `get_weather` pero responde sin usar el resultado | Mas dificil -- logear para revision manual |
| **Tool chaining incorrecto** | Claude usa el output de tool A como input de tool B de forma incorrecta | Validacion de tipos en `execute_tool` |

### Modo debugging interactivo

Para desarrollo, es util poder **pausar** entre steps e inspeccionar el estado:

In [ ]:
async def agent_loop_debug(user_message: str, tools: list):
    """Agent loop con pausas para debugging interactivo."""
    messages = [{"role": "user", "content": user_message}]

    for iteration in range(10):
        response = await client.messages.create(
            model=MODEL, max_tokens=4096,
            tools=tools, messages=messages,
        )

        log_agent_step(iteration + 1, response)

        if response.stop_reason == "end_turn":
            return extract_text(response)

        # Pausa interactiva
        action = input("\n  [Debug] Continuar? (y/n/inspect): ").strip().lower()

        if action == "n":
            return "Abortado por el usuario."
        elif action == "inspect":
            print(f"\n  Messages ({len(messages)} entries):")
            for i, m in enumerate(messages):
                print(f"    [{i}] role={m['role']}, "
                      f"content_type={type(m['content']).__name__}")
            input("  Press Enter para continuar...")

        # Ejecutar tools y continuar
        messages.append({"role": "assistant", "content": response.content})
        tool_results = process_tool_calls(response)
        messages.append({"role": "user", "content": tool_results})

---

## 6.5 Agent Loop Completo y Robusto

Ahora combinamos **todo** -- condiciones de parada, error handling, logging, loop detection, y metricas -- en una implementacion lista para produccion.

### Configuracion y resultado como dataclasses

Usamos `dataclass` para hacer la configuracion y el resultado tipados y claros:

In [ ]:
from dataclasses import dataclass, field
from typing import Any
import time
import json


@dataclass
class AgentConfig:
    """Configuracion del agente."""
    model: str = "claude-sonnet-4-20250514"
    max_iterations: int = 10
    max_token_budget: int = 50_000
    max_consecutive_errors: int = 3
    timeout_seconds: int = 60
    verbose: bool = True


@dataclass
class AgentResult:
    """Resultado de una ejecucion del agente."""
    answer: str
    iterations: int
    total_tokens: int
    total_cost: float
    elapsed_seconds: float
    tool_calls: list[dict] = field(default_factory=list)
    errors: list[str] = field(default_factory=list)
    stopped_reason: str = "completed"

Observaciones sobre las dataclasses:

- **`AgentConfig`** centraliza todos los limites en un solo lugar. En produccion puedes tener configs diferentes por tipo de tarea (ej. una consulta simple usa `max_iterations=5`, un analisis complejo usa `max_iterations=15`).
- **`AgentResult`** captura **todo** lo que necesitas para observabilidad: respuesta, metricas, errores, y la razon por la que se detuvo. Esto es clave para monitoreo en produccion.
- **`stopped_reason`** puede ser: `"completed"`, `"timeout"`, `"token_budget"`, `"max_iterations"`, o `"error_threshold"`. Util para dashboards y alertas.

### La funcion `run_agent` completa

Esta es la implementacion final que integra todas las protecciones:

In [ ]:
async def run_agent(
    message: str,
    tools: list,
    tool_executor,
    config: AgentConfig = AgentConfig(),
) -> AgentResult:
    """
    Agent loop robusto con todas las protecciones.

    Args:
        message: Pregunta del usuario
        tools: Lista de definiciones de tools (JSON Schema)
        tool_executor: Funcion que ejecuta tools: (name, input) -> dict
        config: Configuracion del agente

    Returns:
        AgentResult con respuesta, metricas, y metadata
    """
    messages = [{"role": "user", "content": message}]
    total_tokens = 0
    total_cost = 0.0
    consecutive_errors = 0
    call_history = []
    tool_call_log = []
    error_log = []
    start = time.time()

    for iteration in range(config.max_iterations):
        # --- Timeout check ---
        elapsed = time.time() - start
        if elapsed > config.timeout_seconds:
            return AgentResult(
                answer="Timeout alcanzado.",
                iterations=iteration,
                total_tokens=total_tokens,
                total_cost=total_cost,
                elapsed_seconds=elapsed,
                tool_calls=tool_call_log,
                errors=error_log,
                stopped_reason="timeout",
            )

        # --- Llamar a Claude ---
        response = await client.messages.create(
            model=config.model,
            max_tokens=4096,
            tools=tools,
            messages=messages,
        )

        # --- Actualizar metricas ---
        step_tokens = response.usage.input_tokens + response.usage.output_tokens
        total_tokens += step_tokens
        # Precios Sonnet: $3/M input, $15/M output
        step_cost = (response.usage.input_tokens / 1e6 * 3.0 +
                     response.usage.output_tokens / 1e6 * 15.0)
        total_cost += step_cost

        if config.verbose:
            log_agent_step(iteration + 1, response)

        # --- Token budget check ---
        if total_tokens > config.max_token_budget:
            return AgentResult(
                answer="Presupuesto de tokens excedido.",
                iterations=iteration + 1,
                total_tokens=total_tokens,
                total_cost=total_cost,
                elapsed_seconds=time.time() - start,
                tool_calls=tool_call_log,
                errors=error_log,
                stopped_reason="token_budget",
            )

        # --- Respuesta final ---
        if response.stop_reason == "end_turn":
            final_text = ""
            for block in response.content:
                if block.type == "text":
                    final_text += block.text

            return AgentResult(
                answer=final_text,
                iterations=iteration + 1,
                total_tokens=total_tokens,
                total_cost=total_cost,
                elapsed_seconds=time.time() - start,
                tool_calls=tool_call_log,
                stopped_reason="completed",
            )

        # --- Tool use ---
        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})

            tool_results = []
            for block in response.content:
                if block.type != "tool_use":
                    continue

                # Loop detection
                sig = f"{block.name}:{json.dumps(block.input, sort_keys=True)}"
                if sig in call_history:
                    error_log.append(f"Loop detected: {sig}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": "Error: llamada repetida detectada. "
                                   "Usa la informacion que ya tienes.",
                        "is_error": True,
                    })
                    continue
                call_history.append(sig)

                # Ejecutar tool
                result = tool_executor(block.name, block.input)
                result_str = json.dumps(result) if isinstance(result, dict) else str(result)

                tool_call_log.append({
                    "iteration": iteration + 1,
                    "tool": block.name,
                    "input": block.input,
                    "output": result_str[:500],
                })

                # Error tracking
                is_error = "error" in (result if isinstance(result, dict) else {})
                if is_error:
                    consecutive_errors += 1
                    error_log.append(f"{block.name}: {result_str[:200]}")
                    if consecutive_errors >= config.max_consecutive_errors:
                        return AgentResult(
                            answer="Demasiados errores consecutivos.",
                            iterations=iteration + 1,
                            total_tokens=total_tokens,
                            total_cost=total_cost,
                            elapsed_seconds=time.time() - start,
                            tool_calls=tool_call_log,
                            errors=error_log,
                            stopped_reason="error_threshold",
                        )
                else:
                    consecutive_errors = 0

                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result_str,
                    "is_error": is_error,
                })

            messages.append({"role": "user", "content": tool_results})

    # --- Max iterations alcanzado ---
    return AgentResult(
        answer="Maximo de iteraciones alcanzado.",
        iterations=config.max_iterations,
        total_tokens=total_tokens,
        total_cost=total_cost,
        elapsed_seconds=time.time() - start,
        tool_calls=tool_call_log,
        errors=error_log,
        stopped_reason="max_iterations",
    )

### Flujo visual de `run_agent`

```
run_agent(message, tools, executor, config)
    |
    v
for iteration in range(max_iterations):
    |
    +-- Timeout check -----> STOP (timeout)
    |
    +-- Llamar a Claude
    |
    +-- Token budget check -> STOP (token_budget)
    |
    +-- end_turn? ---------> STOP (completed)    <-- caso ideal
    |
    +-- tool_use?
    |     |
    |     +-- Para cada tool_use block:
    |     |     |
    |     |     +-- Loop detection? --> is_error: True
    |     |     |
    |     |     +-- Ejecutar tool
    |     |     |
    |     |     +-- Error? --> consecutive_errors++
    |     |     |               |
    |     |     |               +-- >= threshold? --> STOP (error_threshold)
    |     |     |
    |     |     +-- Exito --> consecutive_errors = 0
    |     |
    |     +-- Append tool_results a messages
    |
    +-- Siguiente iteracion
    |
    v
STOP (max_iterations)
```

> **Nota sobre el costo**: los precios hardcodeados (`$3/M input`, `$15/M output`) son para Sonnet. Si usas otro modelo, actualiza estos valores. En produccion, es mejor leerlos de una config o de la API.

---

## Resumen de Conceptos Clave -- Semana 6

### Patron ReAct

```
Reason -> Act -> Observe -> Reason -> Act -> Observe -> ... -> Respond
```

- Claude **alterna entre pensar y usar tools**. El flujo es dinamico, no predefinido.
- El agent loop es un `while`/`for` que itera hasta que `stop_reason == "end_turn"`.
- **Messages acumulan** -- cada step agrega `assistant` + `tool_results`. Los tokens crecen linealmente.

### Condiciones de parada

| Condicion | Mecanismo | Cuando aplica |
|-----------|-----------|---------------|
| **Natural stop** | `stop_reason == "end_turn"` | Claude decide que tiene suficiente info |
| **Max iterations** | `for i in range(N)` | Limite duro de pasos |
| **Token budget** | Acumular `usage.input_tokens + output_tokens` | Limite de costo |
| **Timeout** | `asyncio.wait_for()` | Limite de tiempo total |
| **Error threshold** | Contador de errores consecutivos | Tools fallando repetidamente |
| **Loop detection** | Historial de `name:json(input)` | Llamadas repetidas identicas |

### Anatomia de messages

| Campo | Contenido |
|-------|-----------|
| `response.content` | Lista mixta de `TextBlock` (razonamiento) y `ToolUseBlock` (tool calls) |
| `tool_result.tool_use_id` | **Debe** coincidir con el `id` del `ToolUseBlock` |
| `tool_result.is_error` | `True` le dice a Claude que la tool fallo -- puede reintentar o usar otra |

### Debugging

- **Loguear cada step**: thinking, tool calls, resultados, stop_reason, tokens.
- **Modo interactivo**: pausar entre steps para inspeccionr estado.
- **`AgentResult`**: dataclass que captura respuesta, iteraciones, tokens, costo, errores, y `stopped_reason`.

### Modelo mental

```
Pregunta simple (1 tool call)     -->  2-3 iteraciones
Pregunta compuesta (multi-tool)   -->  3-5 iteraciones
Tarea compleja (cadena de tools)  -->  5-10 iteraciones
Algo anda mal (loops/errores)     -->  hits a condition de parada
```